# Test pipeline

Notebook minimale, senza calcoli pesanti, che genera output nello **stesso formato**
degli altri notebook (checkpoint CSV, summary CSV, manifest JSON, figura salvata su
file) e stampa log/progress bar. Serve per testare l'intera catena
notebook -> `.py` -> job PBS -> recupero risultati, senza aspettare ore.


In [ ]:
from pathlib import Path
from time import perf_counter
import json

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import matplotlib
matplotlib.use("Agg")  # backend non interattivo: nessun display sul cluster
import matplotlib.pyplot as plt

print("Test pipeline: avvio")


In [ ]:
# Scrive nella DIRECTORY CORRENTE: chi lancia questo script (submit_chain.sh
# sul cluster, o le istruzioni del README in locale) crea gia' una cartella
# contenitore univoca (nome_timestamp) e ci fa cd DENTRO prima di eseguire
# questo script. Il notebook non deve occuparsi ne' di nome ne' di timestamp:
# solo di scrivere i propri file relativi alla cwd.
OUTPUT_DIR = Path(".")

CHECKPOINT = OUTPUT_DIR / "checkpoint.csv"
SUMMARY_FILE = OUTPUT_DIR / "summary.csv"
MANIFEST_FILE = OUTPUT_DIR / "manifest.json"
PLOT_FILE = OUTPUT_DIR / "test_plot.png"


In [ ]:
# --- Simula un piccolo carico di "lavoro", con log e barra di avanzamento ---
# (utile per verificare che stdout/tqdm finiscano correttamente in logs/*.log)
rng = np.random.default_rng(seed=0)

t0 = perf_counter()
rows = []
N_STEPS = 20
for step in tqdm(range(N_STEPS), desc="test steps"):
    value = rng.normal(loc=0.0, scale=1.0)
    rows.append({"step": step, "value": value})
elapsed = perf_counter() - t0

print(f"Completati {N_STEPS} step in {elapsed:.3f} s")


In [ ]:
# --- Checkpoint CSV (dati grezzi), come negli altri notebook ---
raw = pd.DataFrame(rows)
raw.to_csv(CHECKPOINT, index=False)
print(f"Scritto checkpoint: {CHECKPOINT}")


In [ ]:
# --- Summary CSV, come negli altri notebook ---
summary_results = pd.DataFrame([{
    "n_steps": N_STEPS,
    "mean": raw["value"].mean(),
    "std": raw["value"].std(),
    "elapsed_seconds": elapsed,
}])
summary_results.to_csv(SUMMARY_FILE, index=False)
print(f"Scritto summary: {SUMMARY_FILE}")


In [ ]:
# --- Manifest JSON, come negli altri notebook ---
manifest = {
    "notebook": "test_pipeline",
    "n_steps": N_STEPS,
    "seed": 0,
    "output_dir": str(OUTPUT_DIR),
}
with open(MANIFEST_FILE, "w") as f:
    json.dump(manifest, f, indent=2)
print(f"Scritto manifest: {MANIFEST_FILE}")


In [ ]:
# --- Figura: SALVATA su file, non mostrata a schermo ---
# Sul cluster non c'e' display: plt.show() da solo non produce nessun file
# recuperabile. Va sempre usato savefig (+ close per liberare memoria).
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(raw["step"], raw["value"], marker="o")
ax.set_xlabel("step")
ax.set_ylabel("value")
ax.set_title("Test pipeline - valori simulati")
fig.tight_layout()
fig.savefig(PLOT_FILE, dpi=150)
plt.close(fig)
print(f"Scritta figura: {PLOT_FILE}")


In [ ]:
print("Test pipeline: completato con successo")
